# CIS 5450 Final Project. Difficulty Concept Location Notebook

**Group project**: NYC FHV Driver Pay & Tipping Analysis (2025 TLC Trip Records)

**Annotated notebook**: `CIS5450_FinalProject_NYC_FHV_DriverPay_and_Tip.ipynb`

This notebook documents the **4 Difficulty concepts** plus **1 bonus** implemented in our annotated notebook. Each section below contains:

1. Concept name and rubric category
2. Why we used it (short justification)
3. Cell links (direct hyperlinks to the relevant cells in our Colab notebook)
4. Where results are reflected in the Conclusion


**Note for the grader❗️❗️❗️**: If a cell hyperlink does not jump exactly to the target cell in Colab, please press Ctrl + F (or Cmd + F on Mac) and search for the keyword **Difficulty** to find the relevant section directly.

## Concept 1: Hyperparameter Tuning (Smarter Methods)

**Rubric category**: *"using smarter hyperparameter tuning methods such as Randomized Search or Bayesian Optimization"*

### Why we used this concept

We apply `RandomizedSearchCV`, explicitly listed by the rubric as a "smarter" tuning method versus exhaustive `GridSearchCV`, to **six model families**: Ridge, XGBRegressor, RandomForestRegressor, LogisticRegression, RandomForestClassifier, and MLPClassifier.

Random search is more sample efficient in high dimensional parameter spaces. XGB alone has 7 hyperparameters: a grid with 5 levels each would require 78,125 fits, whereas we cover the same space with 10 to 20 random samples drawn from continuous distributions.

The search uses 3 fold cross validation on training data only, and the held out test set is used **exactly once** for final reporting, preventing CV to test leakage. The MLP decision threshold is also selected on the training subsample alone (max F1).

### Cell links

FROM [**Section header for Hyperparameter Tuning**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=5b-intro-md&line=1&uniqifier=1) TO [**End of Hyperparameter Tuning**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=5b-justify-md&line=1&uniqifier=1)

### Where results are reflected in Conclusion

[**Summry of Hyperparameter Tuning**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=6mTfp53VY9y9). CV tuned 5b RMSE and PR AUC results compared side by side with hand tuned 5a results. For example, CV tuned XGBRegressor reaches RMSE 6.020 versus hand tuned 5a RMSE 5.910, illustrating the subsample versus full train trade off.


## Concept 2: Ensemble Models

**Rubric category**: *"implementing ensemble models"*

### Why we used this concept

We deliberately implement **two distinct ensemble paradigms** for both regression and classification: **bagging** (Random Forest) and **boosting** (XGBoost), rather than using non ensemble alternatives like SVM or KNN.

EDA showed clear non linearities (the airport surcharge step in `driver_pay` versus `trip_miles`) and categorical interactions (operator times hour of day) that purely additive linear models cannot capture. Bagging reduces variance via independent trees; boosting reduces bias via sequential error correction. Comparing their relative performance therefore tells us something about the underlying problem structure.

Each ensemble is wrapped in the same leakage safe `Pipeline` (`ColumnTransformer` plus model) used by the linear baselines for a fair comparison on the identical held out test set.

### Cell links

FROM [**Section header for Extended Models**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=25eca02e&line=11&uniqifier=1) TO [**End of Ensemble Models**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=8a1b1037&line=1&uniqifier=1)

### Where results are reflected in Conclusion

[**Summary of Ensemble Models**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=R6On0sE9ZnYo). XGB hand tuned achieves RMSE 5.910 versus linear baseline RMSE 6.520 (about 10 percent reduction), R squared climbs from 0.864 to 0.889. Both ensemble types significantly outperform the linear baseline. Discussion of why MLP edges out tree ensembles on classification (non additive interactions) and why we would still recommend RF for deployment (interpretability trade off).


## Concept 3: Hypothesis Testing (Simulation Based)

**Rubric category**: *"using simulation based hypothesis testing and statistical analysis to test interpret models and test claims about model validity"*

### Why we used this concept

We conduct **two simulation based permutation tests** (1000 iterations each) to validate model assumptions, exactly as covered in CIS 5450 lectures.

We chose simulation based tests over parametric alternatives (such as t tests) because with about 6 million rows, parametric tests yield essentially zero p values for trivial effects, making it impossible to distinguish economically meaningful results from statistical artifacts. Permutation tests directly simulate the null distribution of our chosen test statistic, making no normality or equal variance assumptions and giving an interpretable answer to *"how unusual is the observed effect under random label assignment?"*

1. **H1**: Trip distance has no effect on driver pay. Tested via permutation on the OLS coefficient of `trip_miles` (two tailed).
2. **H2**: Peak hour fares are equal to off peak fares. Tested via permutation on the difference in mean fare (one tailed, since H1 specifies "higher than").

### Cell links

FROM [**Section header for Hypothesis Testing**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=lhrA1SvvPYs4&line=1&uniqifier=1) TO [**End of Hyperparameter Tuning**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=77058b95-a872-4488-b86f-0d75cd8b5a13&line=9&uniqifier=1)

### Where results are reflected in Conclusion

[**Summry of Hypothesis Testing**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=NTUAsFX8aK1A&line=1&uniqifier=1). Both p values cited verbatim:

1. H1: observed coef 2.86, p value approximately 0.0000 (reject H0).
2. H2: observed difference 0.05 dollar, p value 0.0020 one tailed (reject H0 statistically, but **economically negligible**, used as a teaching moment that statistical significance does not equal practical significance at this data scale).


## Concept 4: Feature Importance (Tree Based)

**Rubric category**: *"using feature importance method in tree models OR using coefficient in regression model to identify most important features"*

### Why we used this concept

We extract `feature_importances_` from the fitted RF and XGB pipelines and visualize the top 15 features per model as horizontal bar charts. Importances are computed as the **mean impurity decrease** contributed by each feature across all trees in each ensemble.

We chose tree based importance over linear coefficient importance because our **best performing models are tree ensembles** (XGB RMSE 5.91 vs LR 6.52). The importances of the worst performing model would be misleading guides to actual feature value. Tree importance also naturally aggregates across hundreds of one hot columns and captures non linear contributions that coefficients cannot represent.

Top features for `driver_pay`: `trip_time` and `trip_miles` together account for about 89 percent of RF importance and 66 percent of XGB importance. Top features for `tipped`: `pickup_hour`, `wait_time_mins`, and LaGuardia airport flags dominate, while `trip_miles` only ranks 7th, supporting the conclusion that **tipping is context driven, not fare driven**.

### Cell links

FROM [**Section header for Feature Importance**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=VZXO6f4Ua0Gy&line=1&uniqifier=1) TO [**End of Feature Importance**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=YPZm4LWN3ezb&line=4&uniqifier=1)

### Where results are reflected in Conclusion

[**Summry of Feature Importance**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=EW6uR59Gm3D0). The top features explicitly drive the Conclusion's stakeholder recommendations.

1. For drivers: `trip_time` plus `trip_miles` dominance supports the recommendation to optimize for trip length maximizing routes (airport queues, off peak long haul).
2. For platforms: tipping classifier's reliance on `wait_time` and `pickup_hour` motivates operational levers (reduce wait, A B test tip prompts at specific hours).
3. Cross model agreement on top 2 features confirms the rankings are robust, not artifacts of a single algorithm.


## Bonus: Permutation Importance (Above and Beyond Cross Method Validation)

**Rubric category**: *"Bonus: Team went above and beyond on one of the concept implementations"* (extending Concept 4 Feature Importance)

### Why this is above and beyond

Tree internal `feature_importances_` from Concept 4 has two well documented weaknesses:

1. Bias toward high cardinality features. One hot `LocationID` columns inherently look "important" because they offer many split point opportunities.
2. Computed on training data. Measures model fit, not whether the feature actually helps the model **generalize**.

To address both weaknesses, we run `sklearn.inspection.permutation_importance` on the **CV tuned XGB regressor** using the **held out test set** (50,000 row sample, 5 repeats). Each feature column is shuffled and the resulting increase in test RMSE is recorded.

This is **model agnostic** and **measured on held out data**, directly answering: *"does this feature actually help generalization?"* Running both methods on the same fitted model and showing they agree on the top features goes substantially beyond what the basic Feature Importance concept requires.

### Cell links

FROM [**Section header for Permutation Importance**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=JypV9rPauC9F&line=1&uniqifier=1) TO [**End of Permutation Importance**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=4YY4ijb1uNLW&line=5&uniqifier=1)

### Where results are reflected in Conclusion

[**Summry of Permutation Importance**](https://colab.research.google.com/drive/1oO2V0EhWLNw32N_I_cTAVBuYvy1bLgex#scrollTo=ZEtIEIUMbM2X&line=1&uniqifier=1). Test set permutation results explicitly cited: shuffling `trip_time` raises RMSE by 10.35; shuffling `trip_miles` raises RMSE by 4.51, confirming the tree based ranking. Conclusion also discusses why methods diverge on LocationID (raw column versus hundreds of one hot columns).
